# **Data Diri**

Nama: Meakhel Gunawan

Email : meakhel220504@gmail.com

# <font color='yellow'> **Import Library**</font>

In [1]:
!pip install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00


In [2]:
!pip install sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.1 MB/s eta 0:00:00


In [3]:
!pip install emoji

In [4]:
# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews, Sort, reviews_all
import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 116
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia
from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

# Library preprocess
import emoji
import csv
import requests
from io import StringIO
from wordcloud import WordCloud

# Library modeling
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import LinearSVC
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D,
    Dense, Dropout, Concatenate, Layer
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import torch

2026-02-04 23:43:11.832319: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770248592.019324      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770248592.070086      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770248592.506903      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770248592.506947      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770248592.506950      55 computation_placer.cc:177] computation placer alr

In [5]:
import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('punkt_tab')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stop words) dalam berbagai bahasa.

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#  <font color='yellow'> **Scapping Dataset**</font>

Dalam proyek ini, dilakukan analisis sentimen terhadap ulasan dan opini pengguna aplikasi Google Gemini yang diperoleh dari Google Play Store. Tujuan utama analisis ini adalah untuk mengidentifikasi persepsi pengguna serta memahami bagaimana pengalaman mereka diekspresikan melalui ulasan yang diberikan.

Melalui penerapan teknik pemrosesan teks (text preprocessing) dan algoritma machine learning dan deep learning, ulasan pengguna akan diklasifikasikan ke dalam tiga kategori sentimen, yaitu positif, negatif, dan netral. Hasil analisis sentimen ini diharapkan mampu memberikan gambaran yang komprehensif mengenai tingkat kepuasan pengguna serta aspek-aspek aplikasi yang perlu dipertahankan atau ditingkatkan.

Dengan demikian, temuan dari analisis ini dapat menjadi sumber wawasan yang bernilai bagi pengembang dalam mengevaluasi dan mengoptimalkan kualitas serta pengalaman penggunaan aplikasi Google Gemini secara berkelanjutan.

In [6]:
# Mengambil semua ulasan dari aplikasi dengan ID 'com.google.android.apps.bard' di Google Play Store.
# Proses scraping mungkin memerlukan beberapa saat tergantung pada jumlah ulasan yang ada.

scrapreview = []
batch_size = 30000  # Ambil data dalam batch kecil
while len(scrapreview) < 90000:
    batch, _ = reviews('com.google.android.apps.bard', lang='id', country='id', sort=Sort.MOST_RELEVANT, count=batch_size)
    scrapreview.extend(batch)

    # Jika jumlah ulasan sudah cukup, hentikan perulangan
    if len(scrapreview) >= 90000:
        scrapreview = scrapreview[:90000]
        break

In [7]:
# Menyimpan ulasan dalam file CSV
app_reviews_df = pd.DataFrame(scrapreview)
app_reviews_df.shape
app_reviews_df.head()
app_reviews_df.to_csv('ulasan_gemini.csv', index=False)